In [1]:
from elasticsearch import Elasticsearch

In [2]:
es = Elasticsearch("http://localhost:9200")

In [4]:
es.info()

ObjectApiResponse({'name': 'b63570a89a7d', 'cluster_name': 'docker-cluster', 'cluster_uuid': '9xep8IsNTbyPCpUf0T5UiQ', 'version': {'number': '8.4.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '42f05b9372a9a4a470db3b52817899b99a76ee73', 'build_date': '2022-10-04T07:17:24.662462378Z', 'build_snapshot': False, 'lucene_version': '9.3.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [5]:
index_name = "course-questions"

In [6]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"}
        }
    }
}

In [ ]:
es.indices.create(
    index=index_name,
    body=index_settings
)

In [7]:
es.indices.create(
    index=index_name,
    body=index_settings
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'course-questions'})

In [8]:
es.indices.get_mapping(index=index_name)

ObjectApiResponse({'course-questions': {'mappings': {'properties': {'course': {'type': 'keyword'}, 'question': {'type': 'text'}, 'section': {'type': 'text'}, 'text': {'type': 'text'}}}}})

In [10]:
from rag import load_faq_documents

documents = load_faq_documents()
len(documents)

1400

In [11]:
for document in documents:
    es.index(index=index_name, document=document)

In [12]:
es.count(index=index_name)

ObjectApiResponse({'count': 1400, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}})

In [13]:
query = "How do I join the course?"

In [14]:
search_query = {
    "query": {
        "multi_match": {
            "query": query,
            "fields": ["question", "text", "section"]
        }
    }
}

In [15]:
response = es.search(
    index=index_name,
    body=search_query
)

In [16]:
response["hits"]["hits"][0]

{'_index': 'course-questions',
 '_id': '3_fzD6EBP9CI-mXYeAWu',
 '_score': 11.840085,
 '_source': {'id': '3f1424af17',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}}

In [17]:
es.indices.get_mapping(index=index_name)

ObjectApiResponse({'course-questions': {'mappings': {'properties': {'answer': {'type': 'text', 'fields': {'keyword': {'type': 'keyword', 'ignore_above': 256}}}, 'course': {'type': 'keyword'}, 'id': {'type': 'text', 'fields': {'keyword': {'type': 'keyword', 'ignore_above': 256}}}, 'question': {'type': 'text'}, 'section': {'type': 'text'}, 'text': {'type': 'text'}}}}})

In [18]:
search_query = {
    "query": {
        "multi_match": {
            "query": query,
            "fields": ["question", "answer", "section"]
        }
    }
}

In [19]:
response = es.search(
    index=index_name,
    body=search_query
)

response["hits"]["hits"][0]


{'_index': 'course-questions',
 '_id': 'LffzD6EBP9CI-mXYbQQE',
 '_score': 12.300481,
 '_ignored': ['answer.keyword'],
 '_source': {'id': 'a11770e750',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I’m new to Slack and can’t find the course channel. Where is it?',
  'answer': "Here’s how you join in Slack: [https://slack.com/help/articles/205239967-Join-a-channel](https://slack.com/help/articles/205239967-Join-a-channel)\n\n- Click “All channels” at the top of your left sidebar. If you don't see this option, click “More” to find it.\n- Browse the list of public channels in your workspace, or use the search bar to search by channel name or description.\n- Select a channel from the list to view it.\n- Click Join Channel.\n\nDo we need to provide the GitHub link to only our code corresponding to the homework questions?\n\nYes. You are required to provide the URL to your repo in order to receive a grade."}}

In [20]:
def elastic_search(query):
    search_query = {
        "query": {
            "multi_match": {
                "query": query,
                "fields": ["question", "answer", "section"]
            }
        }
    }

    response = es.search(
        index=index_name,
        body=search_query
    )

    return [hit["_source"] for hit in response["hits"]["hits"]]

In [21]:
results = elastic_search("How do I join the course?")

In [22]:
results[0]

{'id': 'a11770e750',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I’m new to Slack and can’t find the course channel. Where is it?',
 'answer': "Here’s how you join in Slack: [https://slack.com/help/articles/205239967-Join-a-channel](https://slack.com/help/articles/205239967-Join-a-channel)\n\n- Click “All channels” at the top of your left sidebar. If you don't see this option, click “More” to find it.\n- Browse the list of public channels in your workspace, or use the search bar to search by channel name or description.\n- Select a channel from the list to view it.\n- Click Join Channel.\n\nDo we need to provide the GitHub link to only our code corresponding to the homework questions?\n\nYes. You are required to provide the URL to your repo in order to receive a grade."}

In [23]:
context = "\n\n".join(
    f"Question: {doc['question']}\nAnswer: {doc['answer']}"
    for doc in results[:5]
)

In [24]:
prompt = f"""
Answer the QUESTION based on the CONTEXT.

QUESTION:
{query}

CONTEXT:
{context}
"""

In [25]:
print(prompt[:1500])


Answer the QUESTION based on the CONTEXT.

QUESTION:
How do I join the course?

CONTEXT:
Question: I’m new to Slack and can’t find the course channel. Where is it?
Answer: Here’s how you join in Slack: [https://slack.com/help/articles/205239967-Join-a-channel](https://slack.com/help/articles/205239967-Join-a-channel)

- Click “All channels” at the top of your left sidebar. If you don't see this option, click “More” to find it.
- Browse the list of public channels in your workspace, or use the search bar to search by channel name or description.
- Select a channel from the list to view it.
- Click Join Channel.

Do we need to provide the GitHub link to only our code corresponding to the homework questions?

Yes. You are required to provide the URL to your repo in order to receive a grade.

Question: Course: Can I still join the course after the start date?
Answer: Yes, even if you don't register, you're still eligible to submit the homework.

Be aware, however, that there will be deadl

In [26]:
from openai import OpenAI
client = OpenAI()

In [27]:
response = client.responses.create(
    model="gpt-4o-mini",
    input=prompt
)

In [28]:
response.output_text

'To join the course, you can follow these steps on Slack:\n\n1. Click “All channels” at the top of your left sidebar (or click “More” if you don\'t see this option).\n2. Browse the list of public channels or use the search bar to find the course channel by name or description.\n3. Select the channel to view it.\n4. Click "Join Channel."\n\nAdditionally, if you wish to submit homework or projects, make sure to do so before the submission deadlines. Even if you join after the course start date, you can still participate as long as the submission form is open.'